# 2-1 LangGraph 이론과 고급 기능

원본 슬라이드: [2-1-langgraph-theory-and-advanced-features.pdf](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf)  
이 노트북은 슬라이드의 개념 순서를 따라가는 학습용 해설과 작은 실험입니다. 원본 PDF는 그대로 보존합니다.

## 학습 목표
- 반복·분기를 그래프로 표현해야 하는 이유를 설명한다.
- Node·Edge·State·Reducer의 역할을 구분한다.
- Checkpointer와 사람 승인 지점을 설계한다.

**실습 방식:** 예제를 먼저 실행해 관찰하고, CodeMate에 개선을 요청한 뒤 생성 결과를 직접 검토합니다. 실제 CodeMate 메뉴·버튼 이름은 강의장 환경에서 확인하세요.


## 슬라이드별 찾아보기

각 쪽의 소제목과 첫 핵심 문구를 찾아가기 위한 표입니다. 본문의 설명·실습과 함께 읽고, 원본의 그림·도표·화면 예시는 쪽 번호를 눌러 확인하세요. PDF에서 수식 또는 그림 안 글자가 추출되지 않는 쪽은 원본 화면을 기준으로 읽습니다.

| PDF 쪽 | 소제목 | 확인할 내용 |
|---|---|---|
| [1쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=1) | 2-1 | LangGraph 정통 이론 & 고급 기능 실습 |
| [2쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=2) | LangGraph의 필요성 | Cyclic Graph와 반복 추론 |
| [3쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=3) | LangGraph의 필요성 | 원본 슬라이드의 그림·배치 확인 |
| [4쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=4) | Cyclic Graph와 반복 추론 | Agent는 한 번의 모델 호출로 끝나지 않고 판단, 도구 사용, 관찰을 반복해야 하는 경우가 많다. |
| [5쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=5) | LangChain과 LangGraph 비교 | LangChain은 LLM, 프롬프트, 도구, 파서를 연결하는 체인 구성에 강하다. |
| [6쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=6) | 그래프 구성 요소 | 원본 슬라이드의 그림·배치 확인 |
| [7쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=7) | Node, Edge, State (1) | LangGraph의 기본 요소는 Node, Edge, State |
| [8쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=8) | Node, Edge, State (2) | Agent 그래프는 상태를 읽고 갱신하는 노드들의 연결로 이해할 수 있다. |
| [9쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=9) | State Schema - TypedDict, Pydantic | State Schema는 그래프가 어떤 데이터를 들고 실행되는지 정의한다. |
| [10쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=10) | Reducer | Reducer는 여러 노드가 같은 상태 필드를 갱신할 때 값을 합치는 규칙이다. |
| [11쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=11) | 안정적인 Agent 실행 | 원본 슬라이드의 그림·배치 확인 |
| [12쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=12) | Checkpointer와 상태 영속화 | Checkpointer는 그래프 실행 상태를 저장해 중단과 재개를 가능하게 하는 기능이다. |
| [13쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=13) | Human-in-the-loop (1) | Human-in-the-loop은 Agent 실행 중 특정 지점에서 사람의 판단을 개입시키는 설계이다. |
| [14쪽](../../../raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf#page=14) | Human-in-the-loop (2) | 사람의 개입 지점은 위험도와 비용을 기준으로 설계한다. |


## 1. 반복 워크플로우 (슬라이드 3–5)

Agent는 판단→도구 실행→관찰을 반복하고, 결과에 따라 재검색·재시도·종료를 선택합니다. LangChain은 실행 요소의 연결에, LangGraph는 **공유 상태를 가진 분기·반복·중단**을 명시하는 데 강점이 있습니다. 반복에는 반드시 종료 조건이 필요합니다.

## 2. Node, Edge, State (슬라이드 6–9)

**Node**는 작업 함수, **Edge**는 다음 이동 경로, **State**는 노드 사이에 전달되는 데이터입니다. 상태 스키마를 정하면 노드 간 입력·출력 계약을 읽기 쉬워집니다. 슬라이드 7의 `StateGraph(AgentState)` 예제는 한 노드가 질문을 읽고 답변 필드를 갱신합니다. 아래는 라이브러리 설치 없이 상태 흐름만 보여주는 학습용 코드입니다.

### 슬라이드의 실제 LangGraph 형태 (슬라이드 7, 9–10)

위 순수 Python `draft(state)`는 LangGraph에서 **Node**로 등록할 수 있습니다. 다음 코드는 슬라이드의 `StateGraph`, `START`, `END`를 연결해 한 번 실행하는 형태입니다. 강의장 `langgraph` 설치를 확인한 뒤 실행하세요.

```python
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class SimpleState(TypedDict):
    question: str
    answer: str

def answer_node(state: SimpleState) -> dict:
    return {"answer": f"질문에 답변: {state['question']}"}

graph = StateGraph(SimpleState)
graph.add_node("answer", answer_node)
graph.add_edge(START, "answer")
graph.add_edge("answer", END)
app = graph.compile()
# app.invoke({"question": "상태란?", "answer": ""})
```

여러 Node가 `messages`를 쓰는 경우 슬라이드 10처럼 `Annotated[list, add_messages]`로 합치는 규칙을 선언합니다. `TypedDict`는 상태의 키·타입을 가볍게 적고, 슬라이드 9의 Pydantic은 기본값과 검증을 줄 때 사용합니다.

In [ ]:
from typing import TypedDict
class AgentState(TypedDict):
    question: str
    answer: str
    attempts: int

def draft(state: AgentState) -> dict:
    return {"answer": f"초안: {state['question']}", "attempts": state["attempts"] + 1}

state: AgentState = {"question": "상태란?", "answer": "", "attempts": 0}
state.update(draft(state))
print(state)

## 3. Reducer (슬라이드 10)

두 노드가 같은 상태 필드를 갱신할 때 **어떻게 합칠지** 정하는 규칙입니다. 메시지는 누적할 수 있지만 단일 답변은 최신 값으로 교체할 수 있습니다. 병렬 노드가 같은 키를 갱신한다면 병합 규칙을 먼저 정의해야 합니다.

In [ ]:
state = {"messages": ["사용자: 질문"], "answer": ""}
updates = [{"messages": ["검색: 문서 A"]}, {"messages": ["검토: 출처 확인"]}]
for update in updates:
    state["messages"].extend(update["messages"])  # 교육용 누적 규칙
print(state["messages"])

**검토:** 위 코드에서 `answer`까지 `extend`하면 왜 문제가 될까요? 실제 LangGraph의 메시지 Reducer인 `add_messages`는 단순 리스트 결합보다 더 많은 규칙을 가질 수 있으므로 PDF 10쪽의 선언을 확인하세요.

## 4. Checkpointer와 Human-in-the-loop (슬라이드 11–14)

Checkpointer는 중간 상태를 저장해 중단·재개·추적에 사용합니다. 사람 개입은 외부 발송·결제·삭제처럼 되돌리기 어려운 행동 앞, 또는 정보가 모자란 지점에 둡니다. 승인 결과도 상태에 기록해 재개 뒤에 같은 판단을 적용해야 합니다. **승인 전에는 실제 외부 동작을 실행하지 않습니다.**

In [ ]:
state = {"draft": "안내 메일 초안", "approved": None, "sent": False}
def maybe_send(current):
    if current["approved"] is not True:
        return "발송 보류: 승인 필요"
    return "승인 확인: 실제 발송 함수 연결 지점"  # 이 예제는 발송하지 않음
print(maybe_send(state))
assert state["sent"] is False

### CodeMate 페어 실습

> 질문을 검색하는 노드, 근거를 확인하는 노드, 답변 노드로 이루어진 작은 LangGraph 예제를 작성해 주세요. `TypedDict` 상태와 종료 조건을 먼저 정의하고, 검색 결과가 비었을 때 재시도는 최대 2회로 제한해 주세요. 외부 발송·삭제는 넣지 마세요.

**리뷰 질문:** 상태 업데이트가 각 노드에서 명확한가요? 재시도 제한이 실제로 작동하나요? 중단 지점의 승인 결과를 상태에 남기나요?

## 정리

1. Edge와 State는 각각 무엇을 나타내나요?
2. 병렬 노드가 같은 필드에 쓸 때 Reducer가 없으면 어떤 문제가 생기나요?
3. 승인 거부 후에도 외부 도구가 실행되지 않는지 어떻게 확인하나요?